# 05 · Baseline preprocessing and characterization setup

Process the eight new baseline TIFFs, save baseline-only Zarr stores, and combine them with Notebook 00's existing stores by date. The original stores are unchanged. The combined `ds_a1` and `ds_a2` are lazy views, so merging does not write another full copy.

Place this notebook beside Notebook 00 and run top to bottom in the same environment. Missing dates are left absent; A1 and A2 may have different dates. Processing can be resumed by rerunning the processing cell.

The reader uses the same exported band labels (`YYYY_MM_DD_variable`), float32 values, fill handling and Blosc compression as Notebook 00. These small rasters are read one day at a time. There is no pilot approval step.


## 1. Imports and paths


In [ ]:
# Only if needed: %pip install "zarr<3" numcodecs rasterio xarray "dask[array]" tqdm
from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd
import rasterio
import xarray as xr
import zarr
from numcodecs import Blosc
from tqdm.auto import tqdm

DATA_DIR = Path("../datasets/VNP46")
OUTPUT_DIR = DATA_DIR / "processed" / "baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

A1_STORE = OUTPUT_DIR / "Haiyan_VNP46A1_baseline.zarr"
A2_STORE = OUTPUT_DIR / "Haiyan_VNP46A2_baseline.zarr"
STORE_PATHS = {"VNP46A1": A1_STORE, "VNP46A2": A2_STORE}
EXISTING_STORE_PATHS = {
    product: DATA_DIR / "processed" / f"Haiyan_{product}.zarr"
    for product in STORE_PATHS
}
COMPRESSOR = Blosc(cname="zstd", clevel=5, shuffle=Blosc.BITSHUFFLE)


## 2. Read the baseline file inventory
Only `_Baseline_` TIFFs are selected. Dates come from their band labels, not filename endpoints.


In [ ]:
source_files = sorted(
    set(DATA_DIR.glob("*_Baseline_*.tif"))
    | set(DATA_DIR.glob("*_Baseline_*.tiff"))
)
if not source_files:
    raise FileNotFoundError(f"No baseline TIFFs found in {DATA_DIR.resolve()}")

manifest_records = []
product_grids = {}
for path in source_files:
    product = "VNP46A1" if "VNP46A1" in path.name else "VNP46A2"
    with rasterio.open(path) as src:
        product_grids.setdefault(product, {
            "height": src.height, "width": src.width,
            "transform": tuple(src.transform)[:6],
            "crs_wkt": src.crs.to_wkt(),
        })
        for band_index in src.indexes:
            tags = src.tags(band_index)
            label = (src.descriptions[band_index - 1] or tags.get("long_name")
                     or tags.get("DESCRIPTION") or tags.get("description"))
            year, month, day, variable = label.strip().split("_", 3)
            manifest_records.append({
                "product": product, "source_file": str(path),
                "source_name": path.name, "band_index": band_index,
                "date": pd.Timestamp(f"{year}-{month}-{day}"),
                "variable": variable, "description": label,
                "tags_json": json.dumps(tags, sort_keys=True),
            })

# Shared export endpoints, if present, use the first file's band.
manifest = pd.DataFrame(manifest_records).drop_duplicates(
    ["product", "date", "variable"], keep="first"
)
manifest.to_csv(OUTPUT_DIR / "source_manifest.csv", index=False)
display(manifest.groupby("product").agg(
    files=("source_name", "nunique"), start=("date", "min"),
    end=("date", "max"), dates=("date", "nunique"),
    variables=("variable", "nunique"),
))


## 3. Create or reopen the baseline stores
Existing baseline stores from the earlier Notebook 05 can be resumed. Only a changed baseline date axis requires attention; no store is automatically deleted. Data arrays retain Notebook 00's `(date, y, x)` layout, NaN missing values and one-day spatial chunks.


In [ ]:
stores = {}
for product, store_path in STORE_PATHS.items():
    product_manifest = manifest[manifest["product"].eq(product)]
    dates = pd.DatetimeIndex(sorted(product_manifest["date"].unique()))
    variables = sorted(product_manifest["variable"].unique())
    grid = product_grids[product]
    height, width = grid["height"], grid["width"]
    a, b, c, d, e, f = grid["transform"]

    if store_path.exists():
        root = zarr.open_group(str(store_path), mode="a")
        if not pd.DatetimeIndex(root["date"][:]).equals(dates):
            raise ValueError(f"{store_path.name}: stored dates differ from these TIFFs.")
        stores[product] = root
        print(f"Resuming {product}: {(root['processed'][:] == 1).sum()} / {len(dates)} dates")
        continue

    root = zarr.open_group(str(store_path), mode="w")
    root.attrs.update({
        "product": product, "title": f"Haiyan {product} baseline raster cube",
        "created_utc": datetime.utcnow().isoformat(), **grid,
        "date_start": str(dates.min().date()), "date_end": str(dates.max().date()),
        "date_count": len(dates), "variables": variables,
        "chunk_shape": [1, 512, 512], "missing_value": "NaN", "scaling_applied": False,
    })
    for name, values in {
        "date": dates.values.astype("datetime64[ns]"),
        "x": c + a * (np.arange(width) + 0.5),
        "y": f + e * (np.arange(height) + 0.5),
    }.items():
        array = root.create_dataset(name, data=values, compressor=None)
        array.attrs["_ARRAY_DIMENSIONS"] = [name]

    array = root.create_dataset(
        "processed", data=np.zeros(len(dates), dtype="u1"),
        fill_value=255, compressor=None,
    )
    array.attrs.update({
        "_ARRAY_DIMENSIONS": ["date"], "flag_values": [0, 1],
        "flag_meanings": "not_processed processed",
    })
    array = root.create_dataset("spatial_ref", shape=(), dtype="i4", compressor=None)
    array[()] = 0
    array.attrs.update({
        "_ARRAY_DIMENSIONS": [], "spatial_ref": grid["crs_wkt"],
        "crs_wkt": grid["crs_wkt"], "GeoTransform": f"{c} {a} {b} {f} {d} {e}",
    })
    for variable in variables:
        rows = product_manifest[product_manifest["variable"].eq(variable)]
        array = root.create_dataset(
            variable, shape=(len(dates), height, width),
            chunks=(1, min(512, height), min(512, width)),
            dtype="f4", fill_value=np.nan, compressor=COMPRESSOR,
            dimension_separator="/",
        )
        array.attrs.update({
            "_ARRAY_DIMENSIONS": ["date", "y", "x"], "grid_mapping": "spatial_ref",
            "source_descriptions": sorted(rows["description"].unique().tolist()),
            "source_tags_example": rows["tags_json"].iloc[0], "scaling_applied": False,
        })
    stores[product] = root
    print(f"Created {product}: {len(dates)} dates")


## 4. Process the new baseline dates
Read each day's bands as float32, convert masked pixels and the original fill values to NaN, and write them to Zarr. No scaling or quality filtering is applied during ingestion. Completed dates are skipped. A date is marked complete only after all its variables are written.


In [ ]:
for product, root in stores.items():
    product_manifest = manifest[manifest["product"].eq(product)]
    date_lookup = {
        pd.Timestamp(date): index for index, date in enumerate(root["date"][:])
    }
    readers = {}
    try:
        for date, rows in tqdm(
            product_manifest.groupby("date", sort=True),
            total=product_manifest["date"].nunique(), desc=product,
        ):
            date_index = date_lookup[pd.Timestamp(date)]
            if root["processed"][date_index] == 1:
                continue

            for source_file, file_rows in rows.groupby("source_file", sort=False):
                if source_file not in readers:
                    readers[source_file] = rasterio.open(source_file)
                block = readers[source_file].read(
                    file_rows["band_index"].astype(int).tolist(),
                    masked=True, out_dtype="float32",
                )
                data = np.asarray(block.filled(np.nan), dtype=np.float32)
                data[(data == -9999) | (data == -999) | ~np.isfinite(data)] = np.nan
                for position, variable in enumerate(file_rows["variable"]):
                    root[variable][date_index, :, :] = data[position]

            root["processed"][date_index] = np.uint8(1)
    finally:
        for reader in readers.values():
            reader.close()

    zarr.consolidate_metadata(str(STORE_PATHS[product]))
    print(f"{product}: {(root['processed'][:] == 1).sum()} / {len(date_lookup)} dates complete")


## 5. Combine with the existing stores
The stores are opened lazily. Tiny coordinate-label rounding differences are matched to the existing grid without resampling raster values. Shared dates retain the existing completed data. Each product retains its own available dates; no daily reindexing or A1/A2 date-matching requirement is imposed.

On future sessions, run **sections 1 and 5** to reopen the combined datasets directly.


In [ ]:
combined = {}
for product, baseline_path in STORE_PATHS.items():
    baseline = xr.open_zarr(
        str(baseline_path), chunks={}, consolidated=False, mask_and_scale=False
    )
    existing = xr.open_zarr(
        str(EXISTING_STORE_PATHS[product]), chunks={},
        consolidated=False, mask_and_scale=False,
    )
    if not np.all(baseline["processed"].values == 1):
        raise RuntimeError(f"{product}: finish baseline processing first.")

    # These exports share the same grid; allow the observed header rounding.
    for coordinate in ("x", "y"):
        np.testing.assert_allclose(
            baseline[coordinate].values, existing[coordinate].values,
            rtol=0, atol=1e-7,
        )
    baseline = baseline.assign_coords(x=existing["x"], y=existing["y"])
    baseline["spatial_ref"] = existing["spatial_ref"]

    # An interrupted older expansion may contain unfinished baseline dates.
    existing_dates = pd.DatetimeIndex(existing["date"].values)
    baseline_dates = pd.DatetimeIndex(baseline["date"].values)
    complete = existing["processed"].values == 1
    if len(existing_dates[~complete].difference(baseline_dates)):
        raise RuntimeError(f"{product}: existing event-period processing is incomplete.")
    existing = existing.isel(date=np.flatnonzero(complete))
    added_dates = baseline_dates.difference(existing["date"].values)
    dataset = xr.concat(
        [baseline.sel(date=added_dates), existing], dim="date",
        data_vars="minimal", coords="minimal", compat="equals",
        join="exact", combine_attrs="override",
    ).sortby("date")
    dataset.attrs = dict(existing.attrs)
    dataset.attrs.pop("manifest_signature", None)
    dataset.attrs.update({
        "date_start": str(pd.Timestamp(dataset["date"].values[0]).date()),
        "date_end": str(pd.Timestamp(dataset["date"].values[-1]).date()),
        "date_count": dataset.sizes["date"],
        "source_stores": [str(baseline_path), str(EXISTING_STORE_PATHS[product])],
    })
    for variable in baseline.attrs["variables"]:
        dataset[variable].attrs = dict(existing[variable].attrs)
        dataset[variable].attrs["source_descriptions"] = sorted(set(
            baseline[variable].attrs.get("source_descriptions", [])
            + existing[variable].attrs.get("source_descriptions", [])
        ))
    combined[product] = dataset
    print(f"{product}: {len(added_dates)} baseline dates added; {dataset.sizes['date']} total dates")

ds_a1 = combined["VNP46A1"]
ds_a2 = combined["VNP46A2"]


## 6. Baseline characterization starting point
Use an explicit baseline window. The default is the new historical interval through **2013-05-11**. The first available record is determined by the exported bands, not the nominal January 1 start.

`dnb_baseline_rq` keeps finite, nonnegative DNB-BRDF with `MQF == 0`. Settlement masks and valid-pixel coverage thresholds can be applied in the characterization that follows. Gap-filled nighttime lights remain a diagnostic comparison. Missingness is retained.


In [ ]:
BASELINE_START = "2012-01-01"
BASELINE_END = "2013-05-11"

ds_a1_baseline = ds_a1.sel(date=slice(BASELINE_START, BASELINE_END))
ds_a2_baseline = ds_a2.sel(date=slice(BASELINE_START, BASELINE_END))

dnb_baseline = ds_a2_baseline["DNB_BRDF_Corrected_NTL"]
mqf_baseline = ds_a2_baseline["Mandatory_Quality_Flag"]
gapfilled_baseline = ds_a2_baseline["Gap_Filled_DNB_BRDF_Corrected_NTL"]
dnb_baseline_rq = dnb_baseline.where(
    (mqf_baseline == 0) & np.isfinite(dnb_baseline) & (dnb_baseline >= 0)
)

print(f"Baseline window: {BASELINE_START} to {BASELINE_END}")
print(f"A1: {ds_a1_baseline.sizes['date']} available dates")
print(f"A2: {ds_a2_baseline.sizes['date']} available dates")
